# House Price Prediction Using Machine Learning 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
train_data = pd.read_csv(r"train.csv")
train_data.head(5)

In [ ]:
train_data.shape

In [ ]:
test_data = pd.read_csv(r"test.csv")
test_data.head(5)

In [ ]:
test_ids = test_data['Id'].values.tolist()

### Data Cleaning

In [ ]:
train_data.columns

In [ ]:
train_data.dtypes

In [ ]:
train_data.describe()

In [ ]:
train_data.isna().sum()

In [ ]:
test_data.isna().sum()

Fill the columns NA vals using data description

In [ ]:
# cols whose na values could require imputation
impute_most_frq_cols = ['MSSubClass','MSZoning','Street','LotShape','LandContour','Utilities','LotConfig','LandSlope',
                        'Neighborhood','Condition1','Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st',
                        'Exterior2nd','ExterQual','ExterCond','Foundation','Heating','HeatingQC','CentralAir','Electrical',
                        'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual','Fireplaces','GarageCars',
                        'PavedDrive','SaleType','SaleCondition', 'Functional']

# cols whose na values can be filled with median
impute_median_cols = ['LotFrontage','LotArea','OverallQual','OverallCond','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF',
                        '1stFlrSF','2ndFlrSF','LowQualFinSF','GrLivArea','TotRmsAbvGrd','GarageArea','WoodDeckSF',
                        'OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch','PoolArea','MiscVal', 
                        'YearBuilt', 'YearRemodAdd', 'MoSold', 'YrSold']


# cols whose na values can be filled with default text values (like 'None' or 'No Garage')
# cols whose na values can be filled with 0
# (GarageYrBlt is kept here because if a house has no garage, a 0 is a safe placeholder before scaling/binarizing)
fill_default_cols = ['Alley','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType',
                        'GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature' ,'GarageYrBlt']

default_map = {
    'Alley' : 'No alley',
    'MasVnrType' : 'None',
    'BsmtQual' : 'No Basement',
    'BsmtCond' : 'No Basement',
    'BsmtExposure' : 'No Basement',
    'BsmtFinType1' : 'No Basement',
    'BsmtFinType2' : 'No Basement',
    'FireplaceQu' : 'No Fireplace',
    'GarageType' : 'No Garage',
    'GarageFinish' : 'No Garage',
    'GarageQual' : 'No Garage',
    'GarageCond' : 'No Garage',
    'GarageYrBlt' : 0,
    'PoolQC' : 'No Pool',
    'Fence' : 'No Fence',
    'MiscFeature' : 'None'
}

Fill Default columns with Default Values

In [ ]:

for col in fill_default_cols:
    default_val = default_map.get(col)
    train_data[col] = train_data[col].fillna(default_val)
    test_data[col] = test_data[col].fillna(default_val)
    print(f"Default Values of {col} filled with {default_val}")
    print(train_data[col].unique())
    print(train_data[col].unique())

### Divide train data into train_test - 90,10 split

In [ ]:
from sklearn.model_selection import train_test_split

x = train_data.drop(columns=['SalePrice'],axis=1)
y = train_data['SalePrice']

x_train , x_test , y_train , y_test = train_test_split(x , y , random_state=42 , test_size= 0.1)

In [ ]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

### Imputation

In [ ]:
from sklearn.impute import SimpleImputer

median_imputer = SimpleImputer(strategy='median')
mode_imputer = SimpleImputer(strategy='most_frequent')

#### Mode Imputation

In [ ]:
for col in impute_most_frq_cols:
    x_train[[col]] = mode_imputer.fit_transform(x_train[[col]])

    x_test[[col]] = mode_imputer.transform(x_test[[col]])
    test_data[[col]] = mode_imputer.transform(test_data[[col]])

#### Median Imputation

In [ ]:
for col in impute_median_cols:
    x_train[[col]] = median_imputer.fit_transform(x_train[[col]])

    x_test[[col]] = median_imputer.transform(x_test[[col]])
    test_data[[col]] = median_imputer.transform(test_data[[col]])

### Find Outliers and any Duplicated Data

#### Find if any duplicate data present

In [ ]:
duplicated_data_xtrain = x_train['Id'].duplicated()
x_train[duplicated_data_xtrain]

In [ ]:
duplicated_data_xtest = x_test['Id'].duplicated()
x_test[duplicated_data_xtest]

In [ ]:
duplicated_data_test_data = test_data['Id'].duplicated()
test_data[duplicated_data_test_data]

In [ ]:
from scipy import stats
numeric_cols = x_train.select_dtypes(['int','float','number'])
categorical_cols = x_train.select_dtypes(['object','category','boolean'])

In [ ]:
numeric_cols.drop(columns=['Id'],axis=1,inplace=True)
numeric_cols.columns

In [ ]:
categorical_cols.columns

#### Z-Scores function

In [ ]:
def find_outliers_ZScores(data,col):
    z_scores = np.abs(stats.zscore(data[col]))
    outliers = data[z_scores > 3]
    
    return outliers

#### IQR function

In [ ]:
def find_outliers_IQR(data , col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR
    
    outliers = data[(data[col] < lower_bound) |  (data[col] > upper_bound)]
    
    return outliers

#### Finding outliers using Z-Score and IQR to understand data 

In [ ]:
def create_EDA_table(data , columns):
    outliers_zscore = []
    outliers_iqr = []
    means = []
    medians = []
    skewness = []
    kurtosis = []
    variances = []
    stds = []

    for col in columns:
        outliers_zscore.append(len(find_outliers_ZScores(data,col)))
        outliers_iqr.append(len(find_outliers_IQR(data,col)))
        means.append(data[col].mean())
        medians.append(data[col].median())
        skewness.append(data[col].skew())
        kurtosis.append(data[col].kurtosis())
        variances.append(data[col].var())
        stds.append(data[col].std())
        
        
    EDA_table = pd.DataFrame({
        'Column' : columns,
        'Mean' : means,
        'Median' : medians,
        'Variance' : variances,
        'Standard Deviation' : stds, 
        'Skewness' : skewness,
        'Distribution Symmetry' : list(map((lambda x : 'Left Skewed' if x < -0.1 else 'Right Skewed' if x > 0.1 else 'Normal Distribution') , skewness)),
        'Kurtosis' : kurtosis,
        'Peak and Tail Type' : list(map((lambda x : "Platykurtic" if x < 0 else "Leptokurtic" if x > 0 else "Mesokurtic") , kurtosis)),
        'No of Outliers Based on Z-Score' : outliers_zscore,
        'No of Outliers Based on IQR' : outliers_iqr
    })
    
    return EDA_table

In [ ]:
EDA_table_train_data = create_EDA_table(data=x_train , columns= numeric_cols.columns)
display(EDA_table_train_data)

### Numerical values data preprocessing

#### Find  Cols with extreamly high skew

In [ ]:
problem_cols_train_data = EDA_table_train_data.query("Skewness > 1 or Skewness < -1")
problem_cols_train_data

#### Perform Scaling on the normal data - Standard Scaler

In [ ]:
normal_cols = x_train[numeric_cols.columns.symmetric_difference(problem_cols_train_data['Column'].values)]

In [ ]:
from sklearn.preprocessing import StandardScaler

for col in normal_cols.columns.tolist():
    scaler = StandardScaler()
    
    x_train[[col]] = scaler.fit_transform(x_train[[col]])
    x_test[[col]] = scaler.transform(x_test[[col]])
    test_data[[col]] = scaler.transform(test_data[[col]])
    
    print(f"Successfully Transformed {col} column!!")

#### Is is possible for zero inflation presence in the problematic columns

In [ ]:
for col in problem_cols_train_data['Column'].values.tolist():
    zero_pct = len(x_train[abs(x_train[col]) == 0]) * 100 / len(x_train) 
    
    print(f"Percentage of zeros present in {col} : {zero_pct:.2f}%")

#### Seperating Zero Inflated Columns

In [ ]:
zero_infl_cols = []
threshold = 50

for col in problem_cols_train_data['Column'].values.tolist():
    zero_pct = len(x_train[abs(x_train[col]) == 0]) * 100 / len(x_train) 
    if zero_pct > threshold:
        zero_infl_cols.append(col)
        
zero_infl_cols

For KitchenAbvGr we would binarise differently by combining multiple claseses in order to binarise data while also ensuring normal shape since it is 1 inflated

#### Performing Binarisation and Scaling - ZeroInflated Columns

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

for col in zero_infl_cols:
    x_train[f"Has_{col}"] = (x_train[col] > 0).astype(int) 
    x_test[f"Has_{col}"] = (x_test[col] > 0).astype(int)
    test_data[f"Has_{col}"] = (test_data[col] > 0).astype(int)
    
    x_train.drop(columns = col , axis = 1 , inplace = True)
    x_test.drop(columns = col , axis = 1 , inplace = True)
    test_data.drop(columns = col , axis = 1 , inplace = True)
    
    #scaling new columns
    x_train[[f"Has_{col}"]] = scaler.fit_transform(x_train[[f"Has_{col}"]])
    x_test[[f"Has_{col}"]] = scaler.transform(x_test[[f"Has_{col}"]])
    test_data[[f"Has_{col}"]] = scaler.transform(test_data[[f"Has_{col}"]])
    
    print(f"{col} has been Binarised Sucessfully and Scaled using Standard Scaler!! ")

#### Performing Binarisation and Scaling - OneInflated Columns - KitchenAbvGr

In [ ]:
x_train["HasMoreThan1_KitchenAbvGr"] = (x_train['KitchenAbvGr'] > 1).astype(int)
x_test["HasMoreThan1_KitchenAbvGr"] = (x_test['KitchenAbvGr'] > 1).astype(int)
test_data["HasMoreThan1_KitchenAbvGr"] = (test_data['KitchenAbvGr'] > 1).astype(int)

x_train.drop(columns = ['KitchenAbvGr'] , axis = 1 , inplace = True)
x_test.drop(columns = ['KitchenAbvGr'] , axis = 1 , inplace = True)
test_data.drop(columns = ['KitchenAbvGr'] , axis = 1 , inplace = True)

#scaling new columns
x_train[["HasMoreThan1_KitchenAbvGr"]] = scaler.fit_transform(x_train[["HasMoreThan1_KitchenAbvGr"]])
x_test[["HasMoreThan1_KitchenAbvGr"]] = scaler.transform(x_test[["HasMoreThan1_KitchenAbvGr"]])
test_data[["HasMoreThan1_KitchenAbvGr"]] = scaler.transform(test_data[["HasMoreThan1_KitchenAbvGr"]])

print("Sucessfully Transformed 'KitchenAbvGr' Column Successfully")

#### Remove Zero Inflated Columns from problematic columns list and also remove KitchenAbvGr as it is 1 inflated.

In [ ]:
non_zero_prob_cols = set(problem_cols_train_data['Column']).symmetric_difference(set(zero_infl_cols).union({'KitchenAbvGr'}))
non_zero_prob_cols

#### Transformation our Train and Test data - Problematic Columns

Since our column could have large number of zeros , boxcox could fail , so we would use yeojhonson transfromation instead

#### Performing Transformation

In [ ]:
from sklearn.preprocessing import PowerTransformer

for col in non_zero_prob_cols:
    yeo_jhonson_transformer = PowerTransformer(method='yeo-johnson',standardize=True)
    
    x_train[[col]] = yeo_jhonson_transformer.fit_transform(x_train[[col]])
    x_test[[col]] = yeo_jhonson_transformer.transform(x_test[[col]]) 
    test_data[[col]] = yeo_jhonson_transformer.transform(test_data[[col]]) 
    
    print(f"Transformation of {col} done sucessfully.")

#### Checking Again

In [ ]:
numeric_cols = x_train.select_dtypes(['int','float','number'])
numeric_cols.drop(columns=['Id'],axis=1,inplace=True)

In [ ]:
new_EDA_train_data_table = create_EDA_table(data=x_train , columns=numeric_cols.columns)
new_EDA_train_data_table

In [ ]:
problem_cols_train_data = new_EDA_train_data_table.query("Skewness > 1 or Skewness < -1")
problem_cols_train_data

#### Final Check on Numerical Columns

In [ ]:
numeric_cols.shape

### Categorical values data exploration and  feature Engineering

In [ ]:
categorical_cols = x_train.select_dtypes(['object','category'])

#### Check Unique values in each columns

Nominal Columns  
1. MSZoning
2. Street
3. Alley
4. Neighborhood
5. BldgType
6. LandContour
7. LotConfig
8. HouseStyle
9. Condition1
10. Condition2
11. RoofStyle
12. RoofMatl
13. Exterior1st
14. Exterior2nd
15. MasVnrType
16. Foundation
17. Heating
18. GarageType
19. MiscFeature
20. SaleType
21. SaleCondition
22. Electrical

Ordinal Columns  
1. LotShape
2. Utilities
3. LandSlope
4. ExterQual
5.  ExterCond
6.  BsmtQual
7.  BsmtCond
8.  BsmtExposure
9.  BsmtFinType1
11. BsmtFinType2
12. HeatingQC
13. KitchenQual
14. Functional
15. FireplaceQu
16. GarageFinish
17. GarageQual
18. GarageCond
19. PavedDrive
20. PoolQC
21. Fence

Binary Columns  
1. CentralAir

In [ ]:
ordinal_cols = ['LotShape','Utilities','LandSlope','ExterQual','ExterCond',
                'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional',
                'FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']

nominal_cols = ['MSZoning','Street','Alley','Neighborhood','LotConfig','BldgType','Condition1','Condition2','HouseStyle','LandContour','RoofStyle','RoofMatl','Exterior1st' ,'Exterior2nd' ,
                'MasVnrType' , 'Foundation' , 'Heating' , 'GarageType' , 'MiscFeature' , 'SaleType' , 'SaleCondition', 'Electrical']

binary_cols = ['CentralAir']

In [ ]:
for col in categorical_cols.columns.values:
    print(f"{col}")
    print(categorical_cols[col].unique())
    print("\n")
    print(categorical_cols[col].value_counts())
    print("\n")

#### Find Noise Columns among all the columns

In [ ]:
def noise_col(data , col):
    threshold_freq_pct = 99
    threshold_diff_pct = 30
    
    max_val_pct = data[col].value_counts(normalize = True , dropna = False).max() * 100
    
    if max_val_pct < threshold_freq_pct:
        print(f"{col} is significant")
        return False
    
    categories = data[col].unique().tolist()
    
    if len(categories) > 1:
        group_by_SalePrice = data.groupby(col)['SalePrice'].mean()
        
        max_mean = group_by_SalePrice.max()
        min_mean = group_by_SalePrice.min()
        
        max_min_diff_pct = (max_mean - min_mean) * 100 / min_mean
        
        if max_min_diff_pct < threshold_diff_pct:
            print(f"{col} does not hold any significance")
            return True
        else:
            print(f"{col} is significant")
            return False
    else:
        print(f"{col} only has 1 value , hence make no impact in the output!!")
        return True


In [ ]:
noise_cols = []

for col in ordinal_cols:
    is_noise = noise_col(train_data , col)
    
    if is_noise:
        noise_cols.append(col)
        
for col in nominal_cols:
    is_noise = noise_col(train_data , col)
    
    if is_noise:
        noise_cols.append(col)
        
for col in binary_cols:
    is_noise = noise_col(train_data , col)
    
    if is_noise:
        noise_cols.append(col)

print(f"Noise Columns are : {noise_cols}")

#### Mapping and Scaling for ordinal Columns

In [ ]:
# MAPS 
# ordinal_cols = ['LotShape','Utilities','LandSlope','ExterQual','ExterCond',
#                 'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional',
#                 'FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']

all_maps = {
'LotShape_map' : {'Reg' : 4 , 'IR1' : 3 , 'IR2' : 2 , 'IR3' : 1},
'Utilities_map' : {'AllPub' : 4 , 'NoSewr' : 3 , 'NoSeWa' : 2 , 'ELO' : 1},
'LandSlope_map' : {'Gtl' : 3 , 'Mod' : 2 , 'Sev' : 1},
'ExterQual_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1},
'ExterCond_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1},
'BsmtQual_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1 , 'No Basement' : 0},
'BsmtCond_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1 , 'No Basement' : 0},
'BsmtExposure_map' : {'Gd' : 4 , 'Av' : 3 , 'Mn' : 2 , 'No' : 1 , 'No Basement' : 0},
'BsmtFinType1_map' : {'GLQ' : 6 , 'ALQ' : 5 , 'BLQ' : 4 , 'Rec' : 3 , 'LwQ' : 2 , 'Unf' : 1 , 'No Basement' : 0},
'BsmtFinType2_map' : {'GLQ' : 6 , 'ALQ' : 5 , 'BLQ' : 4 , 'Rec' : 3 , 'LwQ' : 2 , 'Unf' : 1 , 'No Basement' : 0},
'HeatingQC_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1},
'KitchenQual_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1},
'Functional_map' : {'Typ' : 7 , 'Min1' : 6 , 'Min2' : 5 , 'Mod' : 4 , 'Maj1' : 3 , 'Maj2' : 2 , 'Sev' : 1 , 'Sal' : 0},
'FireplaceQu_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1 , 'No Fireplace' : 0},
'GarageFinish_map' : {'Fin' : 3 , 'RFn' : 2 , 'Unf' : 1 , 'No Garage' : 0},
'GarageQual_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1 , 'No Garage' : 0},
'GarageCond_map' : {'Ex' : 5 , 'Gd' : 4 , 'TA' : 3 , 'Fa' : 2 , 'Po' : 1 , 'No Garage' : 0},
'PavedDrive_map' : {'Y' : 3 , 'P' : 2 , 'N' : 1},
'PoolQC_map' : {'Ex' : 4 , 'Gd' : 3 , 'TA' : 2 , 'Fa' : 1 , 'No Pool' : 0},
'Fence_map' : {'GdPrv' : 4 , 'MnPrv' : 3 , 'GdWo' : 2 , 'MnWw' : 1 , 'No Fence' : 0}
}

In [ ]:
# Mapping
for col in ordinal_cols:
    x_train[f"{col}_map"] = x_train[col].map(all_maps[f"{col}_map"])
    x_test[f"{col}_map"] = x_test[col].map(all_maps[f"{col}_map"])
    test_data[f"{col}_map"] = test_data[col].map(all_maps[f"{col}_map"])
    
    print(f"Mapping of {col} Completed in x_train , x_test and test_data!")

In [ ]:
#Scaling of mapped columns
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
#fit_transform -> x_train , transform -> x_test and test_data
for col in ordinal_cols:
    x_train[[f"{col}_map"]] = scaler.fit_transform(x_train[[f"{col}_map"]])
    x_test[[f"{col}_map"]] = scaler.transform(x_test[[f"{col}_map"]])
    test_data[[f"{col}_map"]] = scaler.transform(test_data[[f"{col}_map"]])
    
    print(f"{col}_map scaled using standard scaler!")

In [ ]:
#Drop these ordinal columns after mapping completed
for col in ordinal_cols:
    x_train.drop(columns=col , axis=1 , inplace=True)
    x_test.drop(columns=col , axis=1 , inplace=True)
    test_data.drop(columns=col , axis=1 , inplace=True)
    print(f"{col} dropped in x_train , x_test and test_data!")

#### Mapping for nominal Columns using one_hot_encoder

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

non_ordinal_columns = nominal_cols + binary_cols

for col in non_ordinal_columns:
    x_train_dummies = pd.get_dummies(x_train[[col]] , drop_first=True).astype(int)
    x_test_dummies = pd.get_dummies(x_test[[col]] , drop_first=True).astype(int)
    test_data_dummies = pd.get_dummies(test_data[[col]] , drop_first=True).astype(int)
    
    #Alignment , ensures that test data does contain equal number of columns as train data
    x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns , fill_value=0)
    test_data_dummies = test_data_dummies.reindex(columns = x_train_dummies.columns , fill_value=0)
    
    #Scaling dummies
    for dum_col in x_train_dummies.columns.tolist():
        x_train_dummies[[dum_col]] = scaler.fit_transform(x_train_dummies[[dum_col]])
        x_test_dummies[[dum_col]] = scaler.transform(x_test_dummies[[dum_col]])
        test_data_dummies[[dum_col]] = scaler.transform(test_data_dummies[[dum_col]])
    
    #Concatenate and drop
    x_train = pd.concat([x_train.drop(col , axis=1 ) , x_train_dummies] , axis=1)
    x_test = pd.concat([x_test.drop(col , axis=1 ) , x_test_dummies] , axis=1)
    test_data = pd.concat([test_data.drop(col , axis=1 ) , test_data_dummies] , axis=1)

    print(f"One Hot Encoding and Scaling for {col} Completed!")

### Final Train Data

#### Drop Id column as it is not required

In [ ]:
x_train.drop(columns=['Id'],axis=1 , inplace=True)
x_test.drop(columns=['Id'], axis=1 , inplace=True)

In [ ]:
x_train.to_csv("x_train.csv",index=False)
x_test.to_csv("x_test.csv", index=False)

In [ ]:
test_data.drop(columns='Id' , axis = 1 , inplace=True , errors='ignore')
test_data.to_csv("test_data.csv",index=False)

### PCA

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

# Fit PCA 
pca = PCA()
temp_pca = pca.fit_transform(x_train) 

# CUMULATIVE explained variance
cumulative_evr = np.cumsum(pca.explained_variance_ratio_)

# Plot 
plt.figure(figsize=(10, 6))

# cumulative variance line
plt.plot(range(1, len(cumulative_evr) + 1), cumulative_evr, marker='o', linestyle='-', markersize=0.5)

# horizontal lines for common thresholds (90% and 95%)
plt.axhline(y=0.90, color='r', linestyle='--', label='90% Variance Threshold')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% Variance Threshold')

plt.title("Cumulative Explained Variance by PCA Components")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.legend(loc='lower right')
plt.grid(True , axis='both' )

plt.show()

In [ ]:
test_data.isna().sum()

From PCA graph we can use 130 components for explaining 90% of our data

In [ ]:
pca = PCA(n_components=130)

x_train_pca = pca.fit_transform(x_train)

x_test_pca = pca.transform(x_test)
test_data_pca = pca.transform(test_data)

pca_cols = [f"PCA_{i}" for i in range(1,131)]

x_train_final = pd.DataFrame(x_train_pca, columns=pca_cols)
x_test_final = pd.DataFrame(x_test_pca, columns=pca_cols)
test_data_final = pd.DataFrame(test_data_pca, columns=pca_cols)

print(f"Original training shape: {x_train.shape}")
print(f"Final PCA training shape: {x_train_final.shape}")

### Models Training and Evaluation

In [ ]:
from sklearn.linear_model import LinearRegression , Lasso , Ridge , ElasticNet
from sklearn.metrics import mean_absolute_error , mean_absolute_percentage_error , mean_squared_error , root_mean_squared_error
from sklearn.preprocessing import PowerTransformer

transformer = PowerTransformer(method='yeo-johnson',standardize=True)

y_train_transformed = transformer.fit_transform(y_train.to_frame())

models = {
    "Linear Regression" : LinearRegression(),
    "Lasso Regression" : Lasso(alpha=1),
    "Ridge Regression" : Ridge(alpha=1),
    "Elastic Net" : ElasticNet(alpha=1 , l1_ratio=0.5)
}

In [ ]:
for modal_name , model in models.items():

    model.fit(x_train_final,y_train_transformed)
    predictions = model.predict(x_test_final)

    transformed_predictions = transformer.inverse_transform(predictions.reshape(-1,1)).flatten()
    
    mae = mean_absolute_error(y_test,transformed_predictions)
    mae_pct = mean_absolute_percentage_error(y_test,transformed_predictions)*100
    mse = mean_squared_error(y_test,transformed_predictions)
    rmse = root_mean_squared_error(y_test,transformed_predictions)
    
    print(f"Mean Absoulute Error for {modal_name} model is : {mae:.2f}")
    print(f"Mean Absoulute Error Percentage for {modal_name} model is : {mae_pct:.2f}%")
    print(f"Mean Square Error for {modal_name} model is : {mse:.2f}")
    print(f"Root Mean Square Error for {modal_name} model is : {rmse:.2f}")
    
    sns.kdeplot(y_test,label = "Actual")
    sns.kdeplot(transformed_predictions , label = "Predicted")
    plt.legend()
    plt.show()
    print("\n")

### Result Output Predictions

In [ ]:
best_model = models["Ridge Regression"]

predictions = best_model.predict(test_data_final)

transformed_predictions = transformer.inverse_transform(predictions.reshape(-1,1))

rounded_predictions = [round(x,4) for x in transformed_predictions.flatten()]

result = pd.DataFrame({
    "Id" : test_ids,
    "SalePrice" : rounded_predictions
})

result.to_csv("result.csv",index=False)